# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [3]:
!python3 -m pip install -qU jq lark langchain langchain-elasticsearch langchain_openai tiktoken

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

/opt/anaconda3/envs/langchain_rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial. 

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [9]:
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_URL = getpass("Elastic URL: ")

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_url=ELASTIC_URL,
    es_api_key=ELASTIC_API_KEY,
    index_name="multiquery-lab-index", #give it a meaningful name,
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [10]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [11]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    None

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400 #define chunk size and chunk overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [13]:
documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name="multiquery-lab-index",
    es_url=ELASTIC_URL,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [15]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the sales team at NASA?', '2. How does the sales team at NASA operate?', '3. What are the responsibilities of the sales team at NASA?']


---- Answer ----

The NASA sales team is a part of the Americas region in the company's sales organization. It is led by two Area Vice-Presidents, Laura Martinez for North America and Gary Johnson for South America. The team is responsible for promoting and selling the company's products and services in the North America and South America markets. They work closely with other departments, such as marketing, product development, and customer support, to ensure the company's products and services are effectively targeted and delivered to customers in these regions.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

In [16]:
ans = chain.invoke("How does NASA collaborate with private companies on space missions?")

print("---- Answer 1 ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What is the nature of the collaboration between NASA and private companies for space missions?', '2. Can you explain the partnership between NASA and private companies in regards to space missions?', '3. How do private companies and NASA work together to achieve successful space missions?']


---- Answer 1 ----

NASA collaborates with private companies on space missions through various partnerships and programs. One example is the Commercial Crew Program, which works with private companies such as SpaceX and Boeing to develop and operate spacecraft for transporting astronauts to and from the International Space Station. NASA also has partnerships with private companies for developing and testing new technologies, such as the Lunar Gateway program which aims to establish a sustainable presence on the moon. Additionally, NASA works with private companies through contracts and grants for research and development projects related to space exploration. These collaborations allow NASA to leverage the expertise and resources of private companies while also supporting the growth and innovation of the commercial space industry.


In [17]:
ans = chain.invoke("What technological innovations developed by NASA are used in everyday life?")
print("---- Answer 2 ----")
print(ans)


INFO:langchain.retrievers.multi_query:Generated queries: ["1. What are some examples of NASA's technological innovations that have been integrated into our daily lives?", "2. Can you provide some instances of NASA's technological advancements that have become a part of our everyday routines?", "3. How has NASA's development of new technologies impacted our daily lives?"]


---- Answer 2 ----

I'm sorry, I don't have enough context to answer this question. Could you provide more information about NASA's technological innovations and their impact on everyday life?


In [18]:
ans = chain.invoke("What ethical considerations arise from using AI in space exploration?")
print("---- Answer 3 ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. How does the use of AI in space exploration raise ethical concerns?', '2. What are the potential ethical implications of incorporating AI into space exploration?', '3. In what ways does the utilization of AI in space exploration present ethical challenges?']


---- Answer 3 ----

As an assistant, I am not able to provide a specific answer to this question as it is not mentioned in the given context. However, some potential ethical considerations that may arise from using AI in space exploration could include the potential for AI to make decisions that could harm humans or the environment, the impact on human employment and job displacement, and the potential for AI to perpetuate biases and discrimination. It is important for organizations to carefully consider and address these ethical concerns when implementing AI technology in space exploration.


In [19]:
ans = chain.invoke("What are the most significant benefits of using multi-query retrieval in AI systems?")
print("---- Answer 4 ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. How does multi-query retrieval improve AI systems?', '2. What advantages does multi-query retrieval offer in AI systems?', '3. What are the key benefits of implementing multi-query retrieval in AI systems?']


---- Answer 4 ----

I'm sorry, I don't have enough context to answer this question. Can you provide more information or a specific source for me to retrieve context from?


### 🧩 Reflection – Multi-Query Retriever

In this part, I tested different questions:
- factual (NASA collaborations, Apollo program)  
- ethical (AI in space)  
- technological (NASA inventions)  
- organizational (teamwork and innovation)

The retriever turned each question into several related queries.  
It gave good answers when the context existed and said *“I don’t know”* when the data wasn’t available — showing it works correctly.

From this, I learned that:
- The Multi-Query Retriever helps find **more complete answers** by reformulating the query  
- It still depends on **the data stored** in the vector database  
- The **LLM can only answer based on the retrieved context**

✅ **Conclusion:** I now understand how multi-query retrieval improves search quality and why good, diverse data is essential.
